# 0.Library

In [ ]:
import pandas as pd
from pathlib import Path
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score
)

import importlib
import sys
sys.path.append('../') 
from features import data_utils as du
from features import data_pipeline as dp
from features import general_func as gf
import constants_data as cd

# Reload
importlib.reload(du)
importlib.reload(dp)
importlib.reload(gf)
importlib.reload(cd)

<module 'constants_data' from '/home/smira/myproject/detection_AD_with_VR_data/src/notebooks/../constants_data.py'>

# 1. Paths and Constants

In [2]:
# Constants and paths
parent_folder = Path("../..") # go 2 folder up= "../.."
df_path = parent_folder / "data" / "produced_csv" / "2.cleaned_features_20_patients.csv"
 
df = pd.read_csv(df_path)

df.head()

,Age,Help_Rating_out_of_5,MoCA_Score,Tutorial_total_reading_time,Tutorial_max_reading_time,Tutorial_intensity_reading_time,Tutorial_total_duration_hover,Tutorial_mean_duration_hover,Tutorial_max_duration_hover,Tutorial_std_duration_hover,...,Memory_Yaw_std,Memory_Pitch_std,Memory_Roll_std,Memory_Yaw_range,Memory_dominant_hand_mean_speed,Memory_not_dominant_hand_mean_speed,Memory_dominant_hand_z_range,Memory_not_dominant_hand_y_range,Gender_Male,dominant_hand_Right
0,73.0,4,28.0,45.03,21.00,0.92,20.64,1.03,2.29,0.74,...,26.23,161.72,177.36,99.05,0.16,0.03,0.82,0.80,0,1
1,59.0,2,27.0,171.99,91.56,0.98,53.05,1.02,13.42,2.25,...,47.08,165.33,172.85,180.39,0.06,0.04,1.02,0.61,0,1
2,82.0,1,26.0,338.75,155.14,0.99,145.79,2.11,20.01,4.20,...,23.27,161.90,162.61,109.38,0.06,0.01,0.98,0.75,1,1
3,75.0,2,27.0,114.78,68.19,0.97,46.90,1.47,13.24,2.97,...,25.75,171.61,101.97,234.99,0.06,0.03,0.50,0.59,1,0
4,62.0,1,27.0,152.90,88.47,0.97,54.33,1.81,11.62,2.66,...,30.07,170.64,169.17,248.93,0.12,0.02,1.05,0.58,1,1


# 2. Basic Pipeline : Predicting MoCA score with 20 patients

## 2.1 Train

In [5]:
model  = LinearRegression()
scaler = StandardScaler()

y = df['MoCA_Score']
df = df.drop(columns=['MoCA_Score'])
x = df

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.4, random_state=42)
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

model.fit(X_train_scaled, y_train)
print(model.coef_, model.intercept_)
y_pred = model.predict(X_test_scaled)

[ 0.03720559 -0.07838107 -0.02192174  0.02392501  0.08656844 -0.05729267
 -0.04612562 -0.05683354 -0.0666506  -0.04341855 -0.10338389 -0.014455
  0.07325523 -0.09122936 -0.02052612  0.00387083  0.02593403  0.01028956
  0.00305996  0.02209332 -0.00249605  0.04427883  0.02913595 -0.06473825
 -0.07209565 -0.08883594  0.06485247 -0.0923528  -0.08288199 -0.06278815
 -0.09469827 -0.07989448 -0.0987232  -0.0924591   0.0905825   0.04546735
  0.11876309 -0.04487934 -0.0141551  -0.00702268  0.10076741 -0.04425569
  0.04868821 -0.06329877  0.02771244  0.05493651  0.01287685  0.02007656
 -0.01645364  0.03884004  0.00610193 -0.00410471  0.09587615 -0.07580028
 -0.04936674 -0.01987801  0.00404475 -0.06746781  0.08480103 -0.04714141
  0.00548203 -0.10478152 -0.08525163 -0.09041205 -0.05652954 -0.15005011
 -0.01142599 -0.03328068 -0.01625506 -0.06747015  0.0223997  -0.01211542
  0.05361403  0.08505476 -0.11001346  0.0391141   0.0100782  -0.04777146
 -0.02756448 -0.03502384 -0.02171592 -0.0857882  -0.0

## 2.2 Evaluation

In [6]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.3f}")
print(f"MSE: {mse:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²: {r2:.3f}")

MAE: 1.533
MSE: 3.411
RMSE: 1.847
R²: -6.042


## 2.3 Test overfitting

In [8]:
# TRAIN performance
y_train_pred = model.predict(X_train_scaled)

train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = root_mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

print("TRAIN")
print(f"MAE: {train_mae:.3f}")
print(f"RMSE: {train_rmse:.3f}")
print(f"R²: {train_r2:.3f}")

TRAIN
MAE: 0.000
RMSE: 0.000
R²: 1.000


In [10]:
y.std()
y.describe()

count    20.000000
mean     26.850000
std       2.601113
min      17.000000
25%      27.000000
50%      27.000000
75%      28.000000
max      30.000000
Name: MoCA_Score, dtype: float64

In [11]:
import numpy as np
print(np.var(y))

6.427499999999999


In [9]:
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_val_score

dummy = DummyRegressor(strategy="mean")

scores = cross_val_score(dummy, x, y, cv=5, scoring="r2")
print(scores)
print(scores.mean())

[-0.0703125  -0.21052632 -0.11428571 -0.16537468 -0.88020833]
-0.28814150808222105
